# Phase 30 — Final Development Validation Benchmark
## One-Shot ROUTER_HOLDOUT Execution — A8/C9 Final Development Validation

**Method:** RA-MedFiLM-BARR
**Population:** ROUTER_HOLDOUT — **FINAL DEVELOPMENT VALIDATION BENCHMARK — NOT TEST**
**Specification:** `ra_medfilm_barr_phase30_final_dev_validation_v1_3`

This notebook is a **pure evidence-replay** document. It performs **no model inference** and reads
**no raw ROUTER_HOLDOUT image or GT**. Every number is read from the frozen Phase30 artifacts.

* Primary claim = **A8 vs global non-inferiority** (lower 95% bootstrap CI > −0.005, strict).
* **No p-value** is computed or adjudicated (CASE P2).
* A7 = **DESCRIPTIVE ONLY**.
* H3 = descriptive only, no numeric success threshold.


## 2. Setup

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(r"D:\Softfors\jup")
AUD = ROOT / "reports" / "audits"
TAB = ROOT / "reports" / "tables"


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


def jload(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))


print("Phase30 reviewer notebook — EVIDENCE REPLAY ONLY (no inference, no image read)")
print("ROUTER_HOLDOUT = FINAL DEVELOPMENT VALIDATION BENCHMARK (NOT TEST)")


Phase30 reviewer notebook — EVIDENCE REPLAY ONLY (no inference, no image read)
ROUTER_HOLDOUT = FINAL DEVELOPMENT VALIDATION BENCHMARK (NOT TEST)


## 3. Population and first-open provenance

In [2]:
pop = jload(AUD / "phase30_execution_population_rebind.json")
marker = jload(AUD / "phase30_first_open_marker.json")
print("population:", pop["population"], "n =", pop["n"], "groups =", pop["groups"])
print("domain counts:", pop["domain_counts"])
print("split manifest SHA :", pop["split_manifest"]["sha256"])
print("G batch manifest SHA:", pop["g_batch_manifest"]["sha256"])
print("bootstrap plan SHA  :", pop["bootstrap_plan"]["sha256"])
print()
print("first-open marker UTC:", marker["utc_timestamp"])
print("pre-open counters    :", marker["pre_open_counters"])
print("marker SHA-256       :", sha256_file(AUD / "phase30_first_open_marker.json"))


population: ROUTER_HOLDOUT n = 875 groups = 628
domain counts: {'BUSI': 90, 'ISIC': 383, 'KUB': 298, 'KVASIR': 104}
split manifest SHA : b5da52d6220ef9b59b4ba70d0b403b38bf5234b897615b6ad55d718114db6555
G batch manifest SHA: fc4c785770c45523fdfd440a464820f78caa6bcedeaa1cbe264e10ce5c4e0088
bootstrap plan SHA  : 59c97ccf6fbea823d38a1fba12d5e81b7e5fb103f31bc7fdfe443fd4971faab1

first-open marker UTC: 2026-10-04T03:15:34+00:00
pre-open counters    : {'ROUTER_HOLDOUT_image_reads': 0, 'ROUTER_HOLDOUT_GT_reads': 0, 'ROUTER_HOLDOUT_outcome_reads': 0, 'TEST_reads': 0}
marker SHA-256       : 9b758dc76cd40604ac4436da834e7088b21376cd7041a9472d5e7ac55809f447


## 4. System under test (A8/C9 fixed Phase26 bundle)

In [3]:
sysb = jload(AUD / "phase30_execution_system_rebind.json")
print("bundle manifest:", sysb["bundle_manifest"]["path"], sysb["bundle_manifest"]["sha256"][:16])
for k, v in sysb["components"].items():
    print(f"  {k:8s} {v['sha256'][:16]}  match={v['match']}")
for k, v in sysb["normalizers"].items():
    print(f"  norm {k:14s} {v['sha256'][:16]}  match={v['match']}")
for k, v in sysb["schemas"].items():
    print(f"  schema {k:12s} {v['sha256'][:16]}  match={v['match']}")


bundle manifest: artifacts/phase26_ra_medfilm_barr_bundle_v1_0/canonical_bundle_manifest.json f7aebb0f646bcc8d
  G        286689fdd5c63ad7  match=True
  R_psi    6d28608a669df497  match=True
  B_phi    8a412b2fe82f225f  match=True
  S_omega  eee9b97b9921e53f  match=True
  norm router         ba37742e96610ec5  match=True
  norm post_safety    e112e18b134c6edc  match=True
  schema router       da76aa34f1400614  match=True
  schema post_safety  3e70e20c541b93de  match=True


## 5. Stage A — image-only inference summary

In [4]:
sa = jload(AUD / "phase30_stage_a_inference_summary.json")
print("n =", sa["n"], "| logical G passes =", sa["logical_G_passes"],
      "| G batch forwards =", sa["G_batch_forwards"])
print("G batch forwards by domain:", sa["G_batch_forwards_by_domain"])
print("G semantics:", sa["G_semantics"])
print()
print("router policy:", sa["router"]["policy"], "| score:", sa["router"]["runtime_score"])
print("n_selected =", sa["n_selected"], "| n_skip =", sa["n_skip"],
      "| realized activation =", round(sa["realized_activation_fraction"], 6))
print("R_psi_exec =", sa["n_Rpsi_exec"], "| S_omega_exec =", sa["n_Somega_exec"])
print("accept =", sa["n_accept"], "| reject =", sa["n_reject"], "| forced_skip =", sa["n_forced_skip"])
print("A7 additional model forwards =", sa["A7_additional_model_forwards"])


n = 875 | logical G passes = 1 | G batch forwards = 111
G batch forwards by domain: {'BUSI': 12, 'ISIC': 48, 'KUB': 38, 'KVASIR': 13}
G semantics: {'device': 'cuda', 'autocast_enabled': True, 'autocast_dtype': 'float16', 'grad_context': 'torch.no_grad()', 'inference_mode': False, 'batch_size': 8, 'sample_order': '(dataset asc, sample_id asc)', 'domain_order': ['BUSI', 'ISIC', 'KUB', 'KVASIR'], 'contiguous_within_domain': True}

router policy: C8_FINAL_ROUTER_RESOURCE_AWARE_UTILITY | score: delta_hat_final - 0.25*C
n_selected = 240 | n_skip = 635 | realized activation = 0.274286
R_psi_exec = 240 | S_omega_exec = 240
accept = 236 | reject = 4 | forced_skip = 0
A7 additional model forwards = 0


## 6. Prediction hash barrier (load-bearing)

In [5]:
bar = jload(AUD / "phase30_prediction_hash_barrier.json")
print("barrier token        :", bar["barrier_token"])
print("n_samples            :", bar["n_samples"])
print("GT reads at hash     :", bar["ROUTER_HOLDOUT_GT_reads_at_hash"])
print("GT loaded anywhere   :", bar["GT_loaded_anywhere"])
print("TEST reads           :", bar["TEST_reads"])
for k, v in bar["prediction_hashes"].items():
    print(f"  {k}: {v[:24]}")
assert bar["barrier_token"] == "PREDICTION_HASH_BARRIER_PASS"
assert bar["ROUTER_HOLDOUT_GT_reads_at_hash"] == 0
print("\nPrediction hash barrier PASS.")


barrier token        : PREDICTION_HASH_BARRIER_PASS
n_samples            : 875
GT reads at hash     : 0
GT loaded anywhere   : False
TEST reads           : 0
  global_prob_f16.npy: 399989807fa636be1d39660d
  router_deep_features_f32.npy: 305998a696ae25979f7e4f22
  router_scalars_f32.npy: 34746f042bdd98e38e02ccc9
  roi_geometry.json: 29a75d804078f34bb33a812a
  bphi_delta_hat_f32.npy: c07e65dd555fa2ae3edb3b74
  bphi_utility_f32.npy: 6f45f54af4c9c97973db0246
  router_selected_bool.npy: 8f2ed6fca18e08dec6586644
  a8_final_prob_f16.npy: 76187a990193d1194e0aac7a
  a7_companion_prob_f16.npy: 9ecb7c0dfc62d23ee088723f
  somega_score_f32.npy: 242de91c99324ba4808802b9
  somega_accept_bool.npy: e27c560c0e4fe462ed5d8cd2
  reports/tables/phase30_router_holdout_predictions.csv: 513ccf2905cbc6f544337b9d

Prediction hash barrier PASS.


## 7. Primary result — A8 vs global non-inferiority

**Primary adjudication = bootstrap non-inferiority CI.**
Criterion = **lower macro 95% CI > −0.005 (strict)**. No p-value.

In [6]:
pr = jload(AUD / "phase30_primary_noninferiority_result.json")
print("estimand        :", pr["estimand"])
print("global macro Dice:", round(pr["global_macro_dice"], 6))
print("A8 macro Dice    :", round(pr["a8_macro_dice"], 6))
print("macro DeltaDice  :", round(pr["macro_delta_dice"], 6))
print("macro 95% CI     :", [round(x, 6) for x in pr["macro_ci"]])
print("margin           :", pr["noninferiority_margin"], "(strict)")
print("PRESERVATION_ASSERTED =", pr["PRESERVATION_ASSERTED"])
print("p_values_computed =", pr["p_values_computed"])
assert pr["p_values_computed"] == 0


estimand        : MACRO_DOMAIN_EQUAL_WEIGHT_MEAN_DICE_DELTA
global macro Dice: 0.928548
A8 macro Dice    : 0.930287
macro DeltaDice  : 0.001739
macro 95% CI     : [0.001117, 0.002405]
margin           : 0.005 (strict)
PRESERVATION_ASSERTED = True
p_values_computed = 0


## 8. Per-domain results (macro may not hide domain harm)

In [7]:
df = pd.read_csv(TAB / "phase30_router_holdout_per_domain_results.csv")
print(df.to_string(index=False))


domain   n  global_dice  a8_dice  a7_dice  delta_dice_a8   ci_low  ci_high  domain_harm_flag
  BUSI  90     0.958856 0.958861 0.958861       0.000005 0.000000 0.000015             False
  ISIC 383     0.935914 0.936901 0.936928       0.000986 0.000329 0.001764             False
   KUB 298     0.849725 0.855669 0.855669       0.005945 0.003635 0.008456             False
KVASIR 104     0.969698 0.969718 0.969718       0.000020 0.000000 0.000060             False


## 9. Router metrics (frozen category semantics)

In [8]:
rm = jload(AUD / "phase30_router_metric_results.json")
for k in ("n_selected", "n_skip", "realized_activation_fraction", "precision_among_selected",
          "beneficial_coverage", "harmful_activation_share", "missed_benefit_rate",
          "mean_true_delta_dice_among_selected", "state_counts"):
    print(f"  {k}: {rm.get(k)}")


  n_selected: 240
  n_skip: 635
  realized_activation_fraction: 0.2742857142857143
  precision_among_selected: 0.24583333333333332
  beneficial_coverage: 1.0
  harmful_activation_share: 0.12916666666666668
  missed_benefit_rate: 0.0
  mean_true_delta_dice_among_selected: 0.00896590356263634
  state_counts: {'SKIP': 635, 'EXECUTE_THEN_ACCEPT': 236, 'EXECUTE_THEN_REJECT': 4}


## 10. A7 descriptive companion (DESCRIPTIVE ONLY)

In [9]:
a7 = jload(AUD / "phase30_a7_descriptive_companion_result.json")
print("role:", a7["role"])
print("A7 macro Dice:", round(a7["a7_macro_dice"], 6), "| A8 macro Dice:", round(a7["a8_macro_dice"], 6))
print("A8 - A7 point difference:", round(a7["a8_minus_a7_point_difference"], 6))
print("A7 harmful-replacement count:", a7["a7_harmful_replacement_count"],
      "| A8:", a7["a8_harmful_replacement_count"],
      "| reduction:", a7["absolute_harmful_replacement_reduction"])
print("uncertainty classification:", a7["descriptive_uncertainty"]["classification"])
print("A7 additional model forwards:", a7["A7_additional_model_forwards"])


role: DESCRIPTIVE_COMPANION_ONLY
A7 macro Dice: 0.930294 | A8 macro Dice: 0.930287
A8 - A7 point difference: -7e-06
A7 harmful-replacement count: 31 | A8: 31 | reduction: 0
uncertainty classification: DESCRIPTIVE_UNCERTAINTY_ONLY
A7 additional model forwards: 0


## 11. H3 descriptive result (no numeric gate)

In [10]:
h3 = jload(AUD / "phase30_h3_descriptive_result.json")
print("status:", h3["status"])
print("A7 harmful:", h3["a7_harmful_replacement_count"], "| A8 harmful:",
      h3["a8_harmful_replacement_count"], "| reduction:", h3["absolute_reduction"])
print("H3_PASS:", h3["H3_PASS"], "| H3_FAIL:", h3["H3_FAIL"],
      "| threshold:", h3["minimum_reduction_threshold"])


status: DESCRIPTIVE_ONLY_NO_NUMERIC_SUCCESS_THRESHOLD
A7 harmful: 31 | A8 harmful: 31 | reduction: 0
H3_PASS: None | H3_FAIL: None | threshold: None


## 12. Analytical compute (ANALYTICAL MAC LOWER BOUND)

In [11]:
cp = jload(AUD / "phase30_compute_results.json")
print("label:", cp["accounting_label"])
print("realized activation fraction:", round(cp["realized_activation_fraction"], 6))
print("A8 analytical average cost:", f"{cp['A8_analytical_average_cost']:.0f}")
print("A7 analytical average cost:", f"{cp['A7_analytical_average_cost']:.0f}")
print("s_omega saves compute:", cp["s_omega_saves_compute"])
print("forbidden labels used:", cp["forbidden_labels_used"])


label: ANALYTICAL MAC LOWER BOUND
realized activation fraction: 0.274286
A8 analytical average cost: 32437902023
A7 analytical average cost: 32437901702
s_omega saves compute: False
forbidden labels used: []


## 13. Post-open immutability + final freeze

In [12]:
imm = jload(AUD / "phase30_post_open_immutability_audit.json")
ff = jload(AUD / "phase30_final_freeze.json")
fc = jload(AUD / "phase30_execution_final_check.json")
print("post-open immutability all_pass:", imm["all_pass"])
print("execution final check:", fc["n_checks"] - fc["n_failed"], "/", fc["n_checks"],
      "failed:", fc["failed"])
print("terminal:", ff["terminal"])
print("ROUTER_HOLDOUT:", ff["ROUTER_HOLDOUT_state"], "| preservation_asserted:",
      ff["preservation_asserted"])


post-open immutability all_pass: True
execution final check: 31 / 31 failed: []
terminal: PHASE30_ONE_SHOT_ROUTER_HOLDOUT_EXECUTION_COMPLETE__A8_C9_FINAL_DEVELOPMENT_VALIDATION_RESULT_FROZEN__PHASE31_ELIGIBLE_NOT_STARTED
ROUTER_HOLDOUT: OPENED_AND_FROZEN | preservation_asserted: True


---
## Summary

* ROUTER_HOLDOUT is the **FINAL DEVELOPMENT VALIDATION BENCHMARK — NOT TEST**.
* One logical G extraction pass (111 batch forwards) over 875 samples, exact frozen order.
* Prediction hash barrier **PASS**; GT reads at hash = 0.
* Primary claim = **A8 vs global non-inferiority** (lower 95% CI > −0.005, strict).
* **No p-value** computed or adjudicated.
* A7 = descriptive only; H3 = descriptive only, no numeric gate.
* TEST reads = 0; training = 0.
